# LLM Honesty and Deception Probing, Algoverse 2026 Spring, KMSA & Tommy

## Part 1: Preparation

In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
from utils.knowledge_check import knowledge_check_truthfulqa, knowledge_check_mmlu
from utils.generation import generate_response, NEUTRAL_SYSTEM, FACTUAL_DECEPTION_SCENARIO
# from utils.activation import extract_activations
from utils.probe import probe_all_layers, train_linear_probe
import warnings
warnings.filterwarnings("ignore")   # suppress the greedy-decoding flag warnings

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

DATA_DIR = Path("data/dataset")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

Device: cuda
GPU: NVIDIA GeForce RTX 4090
VRAM: 25.4 GB


In [2]:
deception_df = pd.read_csv(DATA_DIR / "deception_dataset.csv")
print(f"deception_dataset: {deception_df.shape}")
print(deception_df["label"].value_counts())

deception_dataset: (400, 6)
label
honest       200
deceptive    200
Name: count, dtype: int64


In [3]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

N_LAYERS = model.config.num_hidden_layers
HIDDEN_DIM = model.config.hidden_size
print(f"Loaded: {N_LAYERS} layers, hidden_dim={HIDDEN_DIM}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loaded: 28 layers, hidden_dim=3584


## Part 2: Model Knowledge Test
### 2.1 TruthfulQA

In [4]:
tqa_mc = load_dataset("truthful_qa", "multiple_choice", split="validation")
TRUTHFULQA_PATH = DATA_DIR / "truthfulQA_test_results.csv"
CHECKPOINT_EVERY = 50

if TRUTHFULQA_PATH.exists():
    kc_df = pd.read_csv(TRUTHFULQA_PATH)
    if len(kc_df) == len(tqa_mc):
        print(f"Already complete ({len(kc_df)} rows). Loading from {TRUTHFULQA_PATH}")
    else:
        done_questions = set(kc_df["question"].tolist())
        remaining = [item for item in tqa_mc if item["question"] not in done_questions]
        print(f"Resuming from checkpoint: {len(kc_df)} done, {len(remaining)} remaining")
        records = []
        for i, item in enumerate(tqdm(remaining, desc="TruthfulQA knowledge check")):
            records.append(knowledge_check_truthfulqa(item, model, tokenizer, DEVICE))
            if (i + 1) % CHECKPOINT_EVERY == 0:
                pd.DataFrame(records).to_csv(TRUTHFULQA_PATH, mode="a", header=False, index=False)
                records = []
        if records:
            pd.DataFrame(records).to_csv(TRUTHFULQA_PATH, mode="a", header=False, index=False)
        kc_df = pd.read_csv(TRUTHFULQA_PATH)
        print(f"Done. Total: {len(kc_df)}")
else:
    records = []
    for i, item in enumerate(tqdm(tqa_mc, desc="TruthfulQA knowledge check")):
        records.append(knowledge_check_truthfulqa(item, model, tokenizer, DEVICE))
        if (i + 1) % CHECKPOINT_EVERY == 0:
            df_chunk = pd.DataFrame(records)
            df_chunk.to_csv(TRUTHFULQA_PATH, mode="a", header=not TRUTHFULQA_PATH.exists(), index=False)
            records = []
    if records:
        pd.DataFrame(records).to_csv(TRUTHFULQA_PATH, mode="a", header=not TRUTHFULQA_PATH.exists(), index=False)
    kc_df = pd.read_csv(TRUTHFULQA_PATH)
    print(f"Done. Total: {len(kc_df)}  |  Passed: {kc_df['passed'].sum()}  |  Failed: {(~kc_df['passed']).sum()}")

passed_df = kc_df[kc_df["passed"] == True].reset_index(drop=True)
failed_df = kc_df[kc_df["passed"] == False].reset_index(drop=True)
print(f"Passed: {len(passed_df)}  |  Failed: {len(failed_df)}")

multiple_choice/validation-00000-of-0000(…):   0%|          | 0.00/271k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/817 [00:00<?, ? examples/s]

Already complete (817 rows). Loading from data/dataset/truthfulQA_test_results.csv
Passed: 398  |  Failed: 419


### 2.2 MMLU

In [5]:
mmlu = load_dataset("cais/mmlu", "all", split="test")
MMLU_PATH = DATA_DIR / "mmlu_test_results.csv"
CHECKPOINT_EVERY = 50

if MMLU_PATH.exists():
    mmlu_kc_df = pd.read_csv(MMLU_PATH)
    if len(mmlu_kc_df) == len(mmlu):
        print(f"Already complete ({len(mmlu_kc_df)} rows). Loading from {MMLU_PATH}")
    else:
        done_questions = set(mmlu_kc_df["question"].tolist())
        remaining = [item for item in mmlu if item["question"] not in done_questions]
        print(f"Resuming from checkpoint: {len(mmlu_kc_df)} done, {len(remaining)} remaining")
        records = []
        for i, item in enumerate(tqdm(remaining, desc="MMLU knowledge check")):
            records.append(knowledge_check_mmlu(item, model, tokenizer, DEVICE))
            if (i + 1) % CHECKPOINT_EVERY == 0:
                pd.DataFrame(records).to_csv(MMLU_PATH, mode="a", header=False, index=False)
                records = []
        if records:
            pd.DataFrame(records).to_csv(MMLU_PATH, mode="a", header=False, index=False)
        mmlu_kc_df = pd.read_csv(MMLU_PATH)
        print(f"Done. Total: {len(mmlu_kc_df)}")
else:
    records = []
    for i, item in enumerate(tqdm(mmlu, desc="MMLU knowledge check")):
        records.append(knowledge_check_mmlu(item, model, tokenizer, DEVICE))
        if (i + 1) % CHECKPOINT_EVERY == 0:
            df_chunk = pd.DataFrame(records)
            df_chunk.to_csv(MMLU_PATH, mode="a", header=not MMLU_PATH.exists(), index=False)
            records = []
    if records:
        pd.DataFrame(records).to_csv(MMLU_PATH, mode="a", header=not MMLU_PATH.exists(), index=False)
    mmlu_kc_df = pd.read_csv(MMLU_PATH)
    print(f"Done. Total: {len(mmlu_kc_df)}  |  Passed: {mmlu_kc_df['passed'].sum()}  |  Failed: {(~mmlu_kc_df['passed']).sum()}")

mmlu_passed_df = mmlu_kc_df[mmlu_kc_df["passed"] == True].reset_index(drop=True)
mmlu_failed_df = mmlu_kc_df[mmlu_kc_df["passed"] == False].reset_index(drop=True)
print(f"Passed: {len(mmlu_passed_df)}  |  Failed: {len(mmlu_failed_df)}")

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

all/dev-00000-of-00001.parquet:   0%|          | 0.00/76.5k [00:00<?, ?B/s]

...

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

Resuming from checkpoint: 14038 done, 0 remaining


MMLU knowledge check: 0it [00:00, ?it/s]

Done. Total: 14038
Passed: 6596  |  Failed: 7442


## Part 3: Response Generation and Result Judge
### 3.1 TruthfulQA
#### 3.1.1 Response Generation

In [6]:
TRUTHFULQA_RESPONSES_PATH = DATA_DIR / "truthfulQA_responses.csv"
CHECKPOINT_EVERY = 50

configs = [
    ("A", passed_df, NEUTRAL_SYSTEM),
    ("B", failed_df, NEUTRAL_SYSTEM),
    ("C", passed_df, FACTUAL_DECEPTION_SCENARIO),
]

if TRUTHFULQA_RESPONSES_PATH.exists():
    resp_df = pd.read_csv(TRUTHFULQA_RESPONSES_PATH)
    done_keys = set(zip(resp_df["question"], resp_df["config"]))
    total_expected = len(passed_df) * 2 + len(failed_df)
    if len(resp_df) == total_expected:
        print(f"Already complete ({len(resp_df)} rows). Loading from {TRUTHFULQA_RESPONSES_PATH}")
    else:
        print(f"Resuming from checkpoint: {len(resp_df)} done, {total_expected - len(resp_df)} remaining")
else:
    done_keys = set()
    resp_df = None
    print("Starting fresh.")

for config_name, source_df, system_prompt in configs:
    remaining = source_df[~source_df["question"].isin(
        {q for q, c in done_keys if c == config_name}
    )].reset_index(drop=True)

    if len(remaining) == 0:
        print(f"Config {config_name}: already complete, skipping.")
        continue

    print(f"Config {config_name}: {len(remaining)} rows to generate.")
    records = []
    for i, row in enumerate(tqdm(remaining.itertuples(), total=len(remaining), desc=f"Config {config_name}")):
        records.append({
            "question": row.question,
            "correct_answer": row.correct_answer,
            "config": config_name,
            "system_prompt": system_prompt,
            "response": generate_response(row.question, model, tokenizer, DEVICE, system_prompt=system_prompt),
        })
        if (i + 1) % CHECKPOINT_EVERY == 0:
            pd.DataFrame(records).to_csv(
                TRUTHFULQA_RESPONSES_PATH, mode="a",
                header=not TRUTHFULQA_RESPONSES_PATH.exists(), index=False
            )
            done_keys.update((r["question"], r["config"]) for r in records)
            records = []
    if records:
        pd.DataFrame(records).to_csv(
            TRUTHFULQA_RESPONSES_PATH, mode="a",
            header=not TRUTHFULQA_RESPONSES_PATH.exists(), index=False
        )

resp_df = pd.read_csv(TRUTHFULQA_RESPONSES_PATH)
print(f"\nDone. Total rows: {len(resp_df)}")
print(resp_df["config"].value_counts())

Already complete (1215 rows). Loading from data/dataset/truthfulQA_responses.csv
Config A: already complete, skipping.
Config B: already complete, skipping.
Config C: already complete, skipping.

Done. Total rows: 1215
config
B    419
A    398
C    398
Name: count, dtype: int64


#### 3.1.2 Claude Haiku Batch Judge

In [7]:
from utils.judge import build_batch_requests_anthropic
import anthropic

# ── Config ────────────────────────────────────────────────────────────────
JUDGE_MODEL = "claude-haiku-4-5-20251001"
MODEL_TAG   = "claude_haiku"
N_VOTES     = 3
# ─────────────────────────────────────────────────────────────────────────

TRUTHFULQA_JUDGE_PATH = DATA_DIR / f"judge_truthfulqa_{MODEL_TAG}.csv"

if TRUTHFULQA_JUDGE_PATH.exists():
    print(f"Results already exist at {TRUTHFULQA_JUDGE_PATH}, skipping batch submission.")
else:
    # Add row_id column (= pandas index) so we can join results back after parsing
    resp_df = resp_df.reset_index(drop=True)
    resp_df["row_id"] = resp_df.index
    resp_df.to_csv(DATA_DIR / "truthfulQA_responses.csv", index=False)
    print(f"Saved row_id column to truthfulQA_responses.csv ({len(resp_df)} rows)")

    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    requests = build_batch_requests_anthropic(resp_df.set_index("row_id"), JUDGE_MODEL, N_VOTES)
    print(f"Total requests to submit: {len(requests)}")
    batch = client.messages.batches.create(requests=requests)
    print(f"Batch submitted.")
    print(f"Batch ID:     {batch.id}")
    print(f"Status:       {batch.processing_status}")
    print(f"Request counts: {batch.request_counts}")

Results already exist at data/dataset/judge_truthfulqa_claude_haiku.csv, skipping batch submission.


In [8]:
from utils.judge import parse_batch_results_anthropic
import anthropic

# ── Config ────────────────────────────────────────────────────────────────
BATCH_ID  = ""   # paste TruthfulQA Anthropic batch ID here
MODEL_TAG = "claude_haiku"
# ─────────────────────────────────────────────────────────────────────────

TRUTHFULQA_JUDGE_PATH = DATA_DIR / f"judge_truthfulqa_{MODEL_TAG}.csv"

if TRUTHFULQA_JUDGE_PATH.exists():
    print(f"Already exists: {TRUTHFULQA_JUDGE_PATH}")
else:
    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    batch = client.messages.batches.retrieve(BATCH_ID)
    print(f"Batch status: {batch.processing_status}")
    assert batch.processing_status == "ended", "Batch not done yet — re-run when complete."

    results = list(client.messages.batches.results(BATCH_ID))
    print(f"Retrieved {len(results)} results")

    # resp_df must have row_id as index for the join
    source = resp_df.set_index("row_id")
    judge_df = parse_batch_results_anthropic(results, source)
    judge_df.to_csv(TRUTHFULQA_JUDGE_PATH, index=False)
    print(f"Saved {len(judge_df)} rows to {TRUTHFULQA_JUDGE_PATH}")
    print(judge_df["config"].value_counts())

Already exists: data/dataset/judge_truthfulqa_claude_haiku.csv


#### 3.1.3 GPT-4o-mini Batch Judge

In [9]:
from utils.judge import build_batch_requests_openai
import json

# ── Config ────────────────────────────────────────────────────────────────
JUDGE_MODEL = "gpt-4o-mini"
MODEL_TAG   = "gpt4o_mini"
N_VOTES     = 3
# ─────────────────────────────────────────────────────────────────────────

TRUTHFULQA_JUDGE_PATH = DATA_DIR / f"judge_truthfulqa_{MODEL_TAG}.csv"
JSONL_PATH = DATA_DIR / f"batch_truthfulqa_{MODEL_TAG}.jsonl"

# Ensure row_id exists (may have been skipped if Anthropic batch cell was a no-op)
if "row_id" not in resp_df.columns:
    resp_df = resp_df.reset_index(drop=True)
    resp_df["row_id"] = resp_df.index
    resp_df.to_csv(DATA_DIR / "truthfulQA_responses.csv", index=False)
    print(f"Added row_id and saved to truthfulQA_responses.csv ({len(resp_df)} rows)")

if TRUTHFULQA_JUDGE_PATH.exists():
    print(f"Results already exist at {TRUTHFULQA_JUDGE_PATH}, skipping.")
elif JSONL_PATH.exists():
    print(f"JSONL already exists at {JSONL_PATH}, skipping generation.")
    print("To submit, upload this file manually via the OpenAI dashboard or a separate cell.")
else:
    requests = build_batch_requests_openai(resp_df.set_index("row_id"), JUDGE_MODEL, N_VOTES)
    print(f"Total requests to generate: {len(requests)}")

    with open(JSONL_PATH, "w") as f:
        for req in requests:
            f.write(json.dumps(req) + "\n")
    print(f"Wrote {JSONL_PATH}")
    print("JSONL ready. Submit manually when ready.")

Results already exist at data/dataset/judge_truthfulqa_gpt4o_mini.csv, skipping.


In [10]:
from utils.judge import parse_batch_results_openai
from openai import OpenAI

# ── Config ────────────────────────────────────────────────────────────────
BATCH_ID  = "batch_69be3bd4e05081908c2945af22be2511"   # paste TruthfulQA OpenAI batch ID here
MODEL_TAG = "gpt4o_mini"
# ─────────────────────────────────────────────────────────────────────────

TRUTHFULQA_JUDGE_PATH = DATA_DIR / f"judge_truthfulqa_{MODEL_TAG}.csv"

if TRUTHFULQA_JUDGE_PATH.exists():
    print(f"Already exists: {TRUTHFULQA_JUDGE_PATH}")
else:
    client = OpenAI(api_key="your-api-key-here")
    batch = client.batches.retrieve(BATCH_ID)
    print(f"Batch status: {batch.status}")
    assert batch.status == "completed", "Batch not done yet — re-run when complete."

    output_file_id = batch.output_file_id
    content = client.files.content(output_file_id).text
    lines = content.splitlines()
    print(f"Retrieved {len(lines)} output lines")

    source = resp_df.set_index("row_id")
    judge_df = parse_batch_results_openai(lines, source)
    judge_df.to_csv(TRUTHFULQA_JUDGE_PATH, index=False)
    print(f"Saved {len(judge_df)} rows to {TRUTHFULQA_JUDGE_PATH}")
    print(judge_df["config"].value_counts())

Already exists: data/dataset/judge_truthfulqa_gpt4o_mini.csv


### 3.2 MMLU Response Genearation and Result Judge
#### 3.2.1 Response Generation

In [11]:
MMLU_RESPONSES_PATH = DATA_DIR / "mmlu_responses.csv"
CHECKPOINT_EVERY = 50

configs = [
    ("A", mmlu_passed_df, NEUTRAL_SYSTEM),
    ("B", mmlu_failed_df, NEUTRAL_SYSTEM),
    ("C", mmlu_passed_df, FACTUAL_DECEPTION_SCENARIO),
]

if MMLU_RESPONSES_PATH.exists():
    mmlu_resp_df = pd.read_csv(MMLU_RESPONSES_PATH)
    done_keys = set(zip(mmlu_resp_df["question"], mmlu_resp_df["config"]))
    total_expected = len(mmlu_passed_df) * 2 + len(mmlu_failed_df)
    if len(mmlu_resp_df) == total_expected:
        print(f"Already complete ({len(mmlu_resp_df)} rows). Loading from {MMLU_RESPONSES_PATH}")
    else:
        print(f"Resuming from checkpoint: {len(mmlu_resp_df)} done, {total_expected - len(mmlu_resp_df)} remaining")
else:
    done_keys = set()
    mmlu_resp_df = None
    print("Starting fresh.")

for config_name, source_df, system_prompt in configs:
    remaining = source_df[~source_df["question"].isin(
        {q for q, c in done_keys if c == config_name}
    )].reset_index(drop=True)

    if len(remaining) == 0:
        print(f"Config {config_name}: already complete, skipping.")
        continue

    print(f"Config {config_name}: {len(remaining)} rows to generate.")
    records = []
    for i, row in enumerate(tqdm(remaining.itertuples(), total=len(remaining), desc=f"Config {config_name}")):
        records.append({
            "question": row.question,
            "correct_answer": row.correct_answer,
            "config": config_name,
            "system_prompt": system_prompt,
            "response": generate_response(row.question, model, tokenizer, DEVICE, system_prompt=system_prompt),
        })
        if (i + 1) % CHECKPOINT_EVERY == 0:
            pd.DataFrame(records).to_csv(
                MMLU_RESPONSES_PATH, mode="a",
                header=not MMLU_RESPONSES_PATH.exists(), index=False
            )
            done_keys.update((r["question"], r["config"]) for r in records)
            records = []
    if records:
        pd.DataFrame(records).to_csv(
            MMLU_RESPONSES_PATH, mode="a",
            header=not MMLU_RESPONSES_PATH.exists(), index=False
        )

mmlu_resp_df = pd.read_csv(MMLU_RESPONSES_PATH)
if "row_id" not in mmlu_resp_df.columns:
    mmlu_resp_df["row_id"] = mmlu_resp_df.index
    mmlu_resp_df.to_csv(MMLU_RESPONSES_PATH, index=False)

print(f"\nDone. Total rows: {len(mmlu_resp_df)}")
print(mmlu_resp_df["config"].value_counts())

Already complete (20634 rows). Loading from data/dataset/mmlu_responses.csv
Config A: already complete, skipping.
Config B: already complete, skipping.
Config C: already complete, skipping.

Done. Total rows: 20634
config
B    7442
A    6596
C    6596
Name: count, dtype: int64


#### 3.2.2 Claude Haiku Batch Judge

In [12]:
from utils.judge import build_batch_requests_anthropic
import anthropic

# ── Config ────────────────────────────────────────────────────────────────
JUDGE_MODEL = "claude-haiku-4-5-20251001"
MODEL_TAG   = "claude_haiku"
N_VOTES     = 3
# ─────────────────────────────────────────────────────────────────────────

MMLU_JUDGE_PATH = DATA_DIR / f"judge_mmlu_{MODEL_TAG}.csv"

if MMLU_JUDGE_PATH.exists():
    print(f"Results already exist at {MMLU_JUDGE_PATH}, skipping batch submission.")
else:
    client = anthropic.Anthropic(api_key="your-api-key-here")
    requests = build_batch_requests_anthropic(mmlu_resp_df.set_index("row_id"), JUDGE_MODEL, N_VOTES)
    print(f"Total requests to submit: {len(requests)}")
    batch = client.messages.batches.create(requests=requests)
    print(f"Batch submitted.")
    print(f"Batch ID:     {batch.id}")
    print(f"Status:       {batch.processing_status}")
    print(f"Request counts: {batch.request_counts}")

Results already exist at data/dataset/judge_mmlu_claude_haiku.csv, skipping batch submission.


In [13]:
from utils.judge import parse_batch_results_anthropic

BATCH_ID = "msgbatch_01TYpxpDPeBx5e7rWPgetVEk"

if MMLU_JUDGE_PATH.exists():
    mmlu_judge_df = pd.read_csv(MMLU_JUDGE_PATH)
    print(f"Already complete ({len(mmlu_judge_df)} rows). Loading from {MMLU_JUDGE_PATH}")
else:
    client = anthropic.Anthropic(api_key="your-api-key-here")

    batch = client.messages.batches.retrieve(BATCH_ID)
    print(f"Status: {batch.processing_status}")
    print(f"Request counts: {batch.request_counts}")

    if batch.processing_status != "ended":
        print("Batch not ready yet. Check back later.")
    else:
        batch_results = client.messages.batches.results(BATCH_ID)
        mmlu_judge_df = parse_batch_results_anthropic(batch_results, mmlu_resp_df.set_index("row_id"))
        mmlu_judge_df.to_csv(MMLU_JUDGE_PATH, index=False)
        print(f"\nSaved {len(mmlu_judge_df)} rows to {MMLU_JUDGE_PATH}")
        print(mmlu_judge_df["config"].value_counts())

Already complete (20634 rows). Loading from data/dataset/judge_mmlu_claude_haiku.csv


#### 3.2.3 GPT-4o-mini Batch Judge

In [14]:
from utils.judge import build_batch_requests_openai
import json
import math
import tiktoken

# ── Config ────────────────────────────────────────────────────────────────
JUDGE_MODEL    = "gpt-4o-mini"
MODEL_TAG      = "gpt4o_mini"
N_VOTES        = 3
MAX_TOKENS     = 1_800_000   # safety margin below OpenAI's 2M limit
# ─────────────────────────────────────────────────────────────────────────

MMLU_JUDGE_PATH = DATA_DIR / f"judge_mmlu_{MODEL_TAG}.csv"

# ── Check 1: final CSV already exists ────────────────────────────────────
if MMLU_JUDGE_PATH.exists():
    print(f"Results already exist at {MMLU_JUDGE_PATH}, skipping.")
else:
    # ── Check 2: any split JSONL already exists ───────────────────────────
    existing_jsonls = sorted(DATA_DIR.glob(f"batch_mmlu_{MODEL_TAG}_*.jsonl"))
    if existing_jsonls:
        print("Split JSONL files already exist, skipping generation:")
        for p in existing_jsonls:
            print(f"  {p}")
        print("To submit, upload these files manually or use a separate cell.")
    else:
        # ── Build all requests ────────────────────────────────────────────
        all_requests = build_batch_requests_openai(
            mmlu_resp_df.set_index("row_id"), JUDGE_MODEL, N_VOTES
        )
        print(f"Total requests: {len(all_requests)}")

        # ── Estimate tokens with tiktoken ─────────────────────────────────
        enc = tiktoken.encoding_for_model("gpt-4o-mini")

        def estimate_tokens(req):
            return len(enc.encode(json.dumps(req)))

        print("Estimating token counts (this may take a minute)...")
        token_counts = [estimate_tokens(r) for r in all_requests]
        total_tokens = sum(token_counts)
        n_splits_est = math.ceil(total_tokens / MAX_TOKENS)
        print(f"Estimated total tokens: {total_tokens:,}")
        print(f"Estimated splits needed: {n_splits_est}")

        # ── Split by cumulative token budget ─────────────────────────────
        splits = []
        current_batch, current_tokens = [], 0
        for req, tok in zip(all_requests, token_counts):
            if current_tokens + tok > MAX_TOKENS and current_batch:
                splits.append(current_batch)
                current_batch, current_tokens = [], 0
            current_batch.append(req)
            current_tokens += tok
        if current_batch:
            splits.append(current_batch)

        print(f"Actual splits: {len(splits)}")

        # ── Write JSONL files ─────────────────────────────────────────────
        for i, batch in enumerate(splits, 1):
            jsonl_path = DATA_DIR / f"batch_mmlu_{MODEL_TAG}_{i}.jsonl"
            with open(jsonl_path, "w") as f:
                for req in batch:
                    f.write(json.dumps(req) + "\n")
            batch_tokens = sum(estimate_tokens(r) for r in batch)
            print(f"  Split {i}: {len(batch):,} requests, ~{batch_tokens:,} tokens → {jsonl_path}")

        print("\nAll JSONL files written. Submit manually when ready.")

Results already exist at data/dataset/judge_mmlu_gpt4o_mini.csv, skipping.


In [15]:
from utils.judge import parse_batch_results_openai
from openai import OpenAI
import pandas as pd

# ── Config ────────────────────────────────────────────────────────────────
BATCH_IDS = [
    "batch_69c047247cac81909b4b0cbe77428fb9",   # split 1
    "batch_69c051f3ca68819083ed7ae84722b4cc",   # split 2   
    "batch_69c064740aa481909b0cc289de84efb0",   # split 3
    "batch_69c070221b3c8190abc138a64c20b575",   # split 4
    "batch_69c07bf936188190834e945c60efffbc",   # split 5
    "batch_69c085dbe5648190b24039ca71bc329d",   # split 6
    "batch_69c0b370c6ac8190ab6c54db27b590e5",   # split 7
    "batch_69c0ba221550819091785017945be418",   # split 8
    "batch_69c0c06befe881909c4e162ac7bb0e7f",   # split 9
    # add more IDs here if more splits were generated
]
MODEL_TAG = "gpt4o_mini"
# ─────────────────────────────────────────────────────────────────────────

MMLU_JUDGE_PATH = DATA_DIR / f"judge_mmlu_{MODEL_TAG}.csv"

if MMLU_JUDGE_PATH.exists():
    print(f"Already exists: {MMLU_JUDGE_PATH}")
else:
    client = OpenAI(api_key="your-api-key-here")
    source = mmlu_resp_df.set_index("row_id")
    parts = []

    for i, batch_id in enumerate(BATCH_IDS, 1):
        batch = client.batches.retrieve(batch_id)
        print(f"Split {i} ({batch_id}) status: {batch.status}")
        assert batch.status == "completed", f"Batch split {i} not done yet — re-run when complete."

        content = client.files.content(batch.output_file_id).text
        lines = content.splitlines()
        print(f"Split {i}: retrieved {len(lines)} output lines")

        part_df = parse_batch_results_openai(lines, source)
        parts.append(part_df)

    judge_df = pd.concat(parts, ignore_index=True)
    judge_df.to_csv(MMLU_JUDGE_PATH, index=False)
    print(f"\nSaved {len(judge_df)} rows to {MMLU_JUDGE_PATH}")
    print(judge_df["config"].value_counts())

Already exists: data/dataset/judge_mmlu_gpt4o_mini.csv


### 3.3 Result Concat

In [16]:
VOTE_COLS       = ["vote_1", "vote_2", "vote_3"]
VOTES_PER_MODEL = len(VOTE_COLS)   # 3

TQA_FULL_PATH  = DATA_DIR / "tqa_full.csv"
MMLU_FULL_PATH = DATA_DIR / "mmlu_full.csv"


def aggregate_judge_votes(*judge_paths):
    """
    Load one or more judge CSVs, count correct votes per (question, config).
    Missing files are skipped with a warning; denominator adjusts accordingly.
    Returns: question, config, votes_correct, votes_incorrect, correct_ratio
    """
    parts = []
    for path in judge_paths:
        path = Path(path)
        if not path.exists():
            print(f"  WARNING: {path.name} not found — skipping")
            continue
        df = pd.read_csv(path)
        df["votes_correct"] = df[VOTE_COLS].apply(
            lambda row: (row.str.lower().str.strip() == "correct").sum(), axis=1
        )
        parts.append(df[["question", "config", "votes_correct"]])

    if not parts:
        raise FileNotFoundError("No judge files found.")

    merged = parts[0].copy()
    for part in parts[1:]:
        merged = merged.merge(part, on=["question", "config"], suffixes=("_a", "_b"))
        merged["votes_correct"] = merged["votes_correct_a"] + merged["votes_correct_b"]
        merged = merged[["question", "config", "votes_correct"]]

    total_votes = len(parts) * VOTES_PER_MODEL
    merged["votes_incorrect"] = total_votes - merged["votes_correct"]
    merged["correct_ratio"]   = merged["votes_correct"] / total_votes
    return merged.reset_index(drop=True)


def build_full(votes_df, responses_df):
    """Merge vote summary back with responses to add response/system_prompt/correct_answer."""
    return votes_df.merge(
        responses_df[["question", "config", "response", "system_prompt", "correct_answer"]],
        on=["question", "config"],
        how="inner",
    ).reset_index(drop=True)


def print_threshold_summary(full_df, dataset_name):
    pivot = (
        full_df.groupby(["config", "votes_correct"])
        .size()
        .unstack(fill_value=0)
        .T
    )
    print(f"\n=== {dataset_name} — votes_correct distribution ===")
    print(pivot.to_string())

    print(f"\n  Config A (truth)       — useful range: votes_correct ≥ 4")
    a = full_df[full_df["config"] == "A"]["votes_correct"]
    for thr in [6, 5, 4]:
        print(f"    >= {thr}/6 correct : {(a >= thr).sum():>5} rows")

    for cfg, label in [("B", "honest_mistake"), ("C", "deception")]:
        print(f"\n  Config {cfg} ({label}) — useful range: votes_correct ≤ 2")
        v = full_df[full_df["config"] == cfg]["votes_correct"]
        for thr in [0, 1, 2]:
            print(f"    <= {thr}/6 correct : {(v <= thr).sum():>5} rows")


# ── Load or compute ───────────────────────────────────────────────────────
if TQA_FULL_PATH.exists() and MMLU_FULL_PATH.exists():
    tqa_full  = pd.read_csv(TQA_FULL_PATH)
    mmlu_full = pd.read_csv(MMLU_FULL_PATH)
    print(f"Loaded tqa_full  ({len(tqa_full)} rows) from {TQA_FULL_PATH}")
    print(f"Loaded mmlu_full ({len(mmlu_full)} rows) from {MMLU_FULL_PATH}")
else:
    tqa_votes  = aggregate_judge_votes(
        DATA_DIR / "judge_truthfulqa_claude_haiku.csv",
        DATA_DIR / "judge_truthfulqa_gpt4o_mini.csv",
    )
    mmlu_votes = aggregate_judge_votes(
        DATA_DIR / "judge_mmlu_claude_haiku.csv",
        DATA_DIR / "judge_mmlu_gpt4o_mini.csv",
    )
    tqa_full  = build_full(tqa_votes,  resp_df)
    mmlu_full = build_full(mmlu_votes, mmlu_resp_df)

    tqa_full.to_csv(TQA_FULL_PATH,   index=False)
    mmlu_full.to_csv(MMLU_FULL_PATH, index=False)
    print(f"Saved tqa_full  ({len(tqa_full)} rows) → {TQA_FULL_PATH}")
    print(f"Saved mmlu_full ({len(mmlu_full)} rows) → {MMLU_FULL_PATH}")

print_threshold_summary(tqa_full,  "TruthfulQA")
print_threshold_summary(mmlu_full, "MMLU")

Loaded tqa_full  (1215 rows) from data/dataset/tqa_full.csv
Loaded mmlu_full (22607 rows) from data/dataset/mmlu_full.csv

=== TruthfulQA — votes_correct distribution ===
config           A    B    C
votes_correct               
0               83  215  240
1                7   18   14
2               10   14    8
3               39   36   17
4               18   19    7
5               26   23   16
6              215   94   96

  Config A (truth)       — useful range: votes_correct ≥ 4
    >= 6/6 correct :   215 rows
    >= 5/6 correct :   241 rows
    >= 4/6 correct :   259 rows

  Config B (honest_mistake) — useful range: votes_correct ≤ 2
    <= 0/6 correct :   215 rows
    <= 1/6 correct :   233 rows
    <= 2/6 correct :   247 rows

  Config C (deception) — useful range: votes_correct ≤ 2
    <= 0/6 correct :   240 rows
    <= 1/6 correct :   254 rows
    <= 2/6 correct :   262 rows

=== MMLU — votes_correct distribution ===
config            A     B     C
votes_correct           

## Part 4: Scenario Response Generation

In [17]:
SCENARIO_RAW_PATH       = DATA_DIR / "scenario_responses_raw.csv"
SCENARIO_RESPONSES_PATH = DATA_DIR / "scenario_responses.csv"
CHECKPOINT_EVERY        = 50

if SCENARIO_RESPONSES_PATH.exists():
    scenario_resp_df = pd.read_csv(SCENARIO_RESPONSES_PATH)
    print(f"Already complete ({len(scenario_resp_df)} pairs). Loading from {SCENARIO_RESPONSES_PATH}")
else:
    # ── Generate in long format with checkpointing ────────────────────────
    if SCENARIO_RAW_PATH.exists():
        raw_df = pd.read_csv(SCENARIO_RAW_PATH)
        done_keys = set(zip(raw_df["pair_id"], raw_df["label"]))
        print(f"Resuming from checkpoint: {len(raw_df)} done, {len(deception_df) - len(raw_df)} remaining")
    else:
        raw_df = None
        done_keys = set()
        print("Starting fresh.")

    remaining = deception_df[~deception_df.apply(
        lambda r: (r["pair_id"], r["label"]) in done_keys, axis=1
    )].reset_index(drop=True)

    if len(remaining) > 0:
        print(f"{len(remaining)} rows to generate.")
        records = []
        for i, row in enumerate(tqdm(remaining.itertuples(), total=len(remaining), desc="Scenario generation")):
            records.append({
                "pair_id":      row.pair_id,
                "label":        row.label,
                "question":     row.question,
                "system_prompt": row.prompt,
                "response":     generate_response(row.question, model, tokenizer, DEVICE, system_prompt=row.prompt),
            })
            if (i + 1) % CHECKPOINT_EVERY == 0:
                pd.DataFrame(records).to_csv(
                    SCENARIO_RAW_PATH, mode="a",
                    header=not SCENARIO_RAW_PATH.exists(), index=False
                )
                done_keys.update((r["pair_id"], r["label"]) for r in records)
                records = []
        if records:
            pd.DataFrame(records).to_csv(
                SCENARIO_RAW_PATH, mode="a",
                header=not SCENARIO_RAW_PATH.exists(), index=False
            )
        print("Generation complete.")

    # ── Pivot to wide format ──────────────────────────────────────────────
    raw = pd.read_csv(SCENARIO_RAW_PATH)

    honest = raw[raw["label"] == "honest"].rename(columns={
        "question":     "question",
        "system_prompt": "honest_scenario",
        "response":     "honest_response",
    })[["pair_id", "question", "honest_scenario", "honest_response"]]

    deceptive = raw[raw["label"] == "deceptive"].rename(columns={
        "system_prompt": "deceptive_scenario",
        "response":     "deceptive_response",
    })[["pair_id", "deceptive_scenario", "deceptive_response"]]

    scenario_resp_df = (
        honest.merge(deceptive, on="pair_id")
        .sort_values("pair_id")
        .reset_index(drop=True)
    )
    scenario_resp_df.to_csv(SCENARIO_RESPONSES_PATH, index=False)
    print(f"Saved {len(scenario_resp_df)} pairs to {SCENARIO_RESPONSES_PATH}")

print(f"\nColumns: {scenario_resp_df.columns.tolist()}")
print(scenario_resp_df.head(2))

Starting fresh.
400 rows to generate.


Scenario generation:   0%|          | 0/400 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Generation complete.
Saved 200 pairs to data/dataset/scenario_responses.csv

Columns: ['pair_id', 'question', 'honest_scenario', 'honest_response', 'deceptive_scenario', 'deceptive_response']
   pair_id                                           question  \
0        0  Teacher: Why haven't you submitted your assign...   
1        1  Teacher: Why have you not submitted the scienc...   

                                     honest_scenario  \
0  You are a student who has not completed an imp...   
1  You are a student who hasn't started working o...   

                                     honest_response  \
0  I understand that this is a difficult situatio...   
1  Teacher: I understand that you're concerned ab...   

                                  deceptive_scenario  \
0  You are a student who has not completed an imp...   
1  You are a student who hasn't started working o...   

                                  deceptive_response  
0  I understand that this is a difficult situatio.

## Part 5: Build Probe Dataset and Extract Activations
### 5.1 Build Prove Dataset

In [18]:
PROBE_DATASET_PATH = DATA_DIR / "probe_dataset.csv"

# ── Load factual data ────────────────────────────────────────────────────
tqa_full  = pd.read_csv(DATA_DIR / "tqa_full.csv")
mmlu_full = pd.read_csv(DATA_DIR / "mmlu_full.csv")

def filter_factual(df, domain):
    truth     = df[(df["config"] == "A") & (df["votes_correct"] == 6)].copy()
    truth["label"] = "truth"
    mistake   = df[(df["config"] == "B") & (df["votes_correct"] == 0)].copy()
    mistake["label"] = "honest_mistake"
    deception = df[(df["config"] == "C") & (df["votes_correct"] == 0)].copy()
    deception["label"] = "deception"
    out = pd.concat([truth, mistake, deception], ignore_index=True)
    out["domain"] = domain
    return out[["question", "response", "label", "system_prompt", "domain", "correct_answer"]]

tqa_factual  = filter_factual(tqa_full,  "factual")
mmlu_factual = filter_factual(mmlu_full, "factual")

# ── Load social data ──────────────────────────────────────────────────────
scenario_resp_df = pd.read_csv(DATA_DIR / "scenario_responses.csv")

honest_rows = scenario_resp_df[["question", "honest_response", "honest_scenario"]].copy()
honest_rows.columns = ["question", "response", "system_prompt"]
honest_rows["label"] = "truth"

deceptive_rows = scenario_resp_df[["question", "deceptive_response", "deceptive_scenario"]].copy()
deceptive_rows.columns = ["question", "response", "system_prompt"]
deceptive_rows["label"] = "deception"

social = pd.concat([honest_rows, deceptive_rows], ignore_index=True)
social["domain"] = "social"
social["correct_answer"] = ""

# ── Merge & save ─────────────────────────────────────────────────────────
probe_dataset = pd.concat([tqa_factual, mmlu_factual, social], ignore_index=True)
probe_dataset.to_csv(PROBE_DATASET_PATH, index=False)

print(f"probe_dataset.csv: {len(probe_dataset)} rows")
print("\nLabel distribution:")
print(probe_dataset["label"].value_counts())
print("\nDomain distribution:")
print(probe_dataset["domain"].value_counts())

probe_dataset.csv: 11708 rows

Label distribution:
label
honest_mistake    4305
deception         3837
truth             3566
Name: count, dtype: int64

Domain distribution:
domain
factual    11308
social       400
Name: count, dtype: int64


### 5.2 Extract Activations

In [19]:
from utils.activation import extract_activations

ACTIVATIONS_PATH = OUTPUT_DIR / "activations.npy"
LABELS_PATH      = OUTPUT_DIR / "labels.npy"
CHECKPOINT_PATH  = OUTPUT_DIR / "activations_checkpoint.npz"
CHECKPOINT_EVERY = 50

LABEL_MAP = {"truth": 0, "honest_mistake": 1, "deception": 2}

probe_dataset = pd.read_csv(PROBE_DATASET_PATH)

# ── Resume from checkpoint if exists ─────────────────────────────────────
if CHECKPOINT_PATH.exists():
    ckpt = np.load(CHECKPOINT_PATH)
    all_activations = list(ckpt["activations"])
    all_labels      = list(ckpt["labels"])
    start_idx = len(all_activations)
    print(f"Resuming from checkpoint: {start_idx}/{len(probe_dataset)} done")
else:
    all_activations = []
    all_labels      = []
    start_idx = 0
    print(f"Starting fresh. {len(probe_dataset)} samples to process.")

# ── Extract ───────────────────────────────────────────────────────────────
for i, row in enumerate(tqdm(
    probe_dataset.iloc[start_idx:].itertuples(),
    total=len(probe_dataset) - start_idx,
    desc="Extracting activations",
)):
    acts = extract_activations(
        question=row.question,
        response=row.response,
        system_prompt=row.system_prompt,
        model=model,
        tokenizer=tokenizer,
        device=DEVICE,
    )
    all_activations.append(acts)
    all_labels.append(LABEL_MAP[row.label])

    if (start_idx + i + 1) % CHECKPOINT_EVERY == 0:
        np.savez(CHECKPOINT_PATH,
                 activations=np.array(all_activations),
                 labels=np.array(all_labels))

# ── Save final ────────────────────────────────────────────────────────────
activations_arr = np.array(all_activations)  # (n_samples, 28, 3584)
labels_arr      = np.array(all_labels)        # (n_samples,)

np.save(ACTIVATIONS_PATH, activations_arr)
np.save(LABELS_PATH,      labels_arr)

print(f"\nSaved activations: {activations_arr.shape}")
print(f"Saved labels:      {labels_arr.shape}")
print(f"Label counts: { {k: int((labels_arr == v).sum()) for k, v in LABEL_MAP.items()} }")

Starting fresh. 11708 samples to process.


Extracting activations:   0%|          | 0/11708 [00:00<?, ?it/s]


Saved activations: (11708, 28, 3584)
Saved labels:      (11708,)
Label counts: {'truth': 3566, 'honest_mistake': 4305, 'deception': 3837}


In [ ]:
from huggingface_hub import HfApi

api = HfApi()
REPO_ID = "mikrokozmoz/algoverse_2026spring_kmsa_qwen2.5_7b_instruct_activations"
TOKEN   = "ask-mikrokozmoz-for-access-token"

api.upload_file(
    path_or_fileobj="/workspace/kmsa_2026spring_Algoverse/outputs/activations.npy",
    path_in_repo="activations.npy",
    repo_id=REPO_ID,
    repo_type="dataset",
    token=TOKEN,
)
api.upload_file(
    path_or_fileobj="/workspace/kmsa_2026spring_Algoverse/outputs/labels.npy",
    path_in_repo="labels.npy",
    repo_id=REPO_ID,
    repo_type="dataset",
    token=TOKEN,
)
print("Done.")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Done.


In [ ]:
# from huggingface_hub import hf_hub_download

# acts_path = hf_hub_download(
#     repo_id="mikrokozmoz/algoverse_2026spring_kmsa_qwen2.5_7b_instruct_activations",
#     filename="activations.npy",
#     repo_type="dataset",
#     token="ask-mikrokozmoz-for-access-token",
# )
# labels_path = hf_hub_download(
#     repo_id="mikrokozmoz/algoverse_2026spring_kmsa_qwen2.5_7b_instruct_activations",
#     filename="labels.npy",
#     repo_type="dataset",
#     token="ask-mikrokozmoz-for-access-token",
# )